# 01 — Source extraction

Single point of entry for every number used by the calibration. One section per
source. Each section holds the extracted values as **plain Python literals** —
manual, transparent extraction with a note pointing at the exact page/cell of
the source. The final cell writes the four observation CSVs into `data/`.

**The literals in this notebook are the master; the CSVs are generated
artifacts** (checked in for diff-ability, regenerated by re-running this
notebook top to bottom). To add a source: register it in the
`SOURCES_REGISTER` cell, add a section, append rows, re-run.

Scope reminder: track access, station access, shunting, parking and traction
energy are **excluded** from this calibration workstream. Their figures may
still be extracted (marked `in_scope = no` in the bridge mapping) because
they are needed to strip infrastructure/energy shares out of published totals.


In [1]:
from pathlib import Path

import pandas as pd


# Anchor data/ next to this notebook, independent of the kernel's working
# directory — PyCharm and Jupyter set different cwds, and a bare relative
# Path("data") breaks as soon as they disagree. Resolution order: cwd is
# the notebooks dir itself; cwd is data/; cwd is backend/ or the repo root.
def _resolve_data_dir() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "01_source_extraction.ipynb").exists():
        return cwd / "data"
    if cwd.name == "data":
        return cwd
    for sub in (
        "models/compositions/notebooks/data",
        "backend/models/compositions/notebooks/data",
        "compositions/notebooks/data",
        "notebooks/data",
    ):
        if (cwd / sub).parent.is_dir():
            return cwd / sub
    return cwd / "data"


DATA_DIR = _resolve_data_dir()
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"data directory: {DATA_DIR}")

# Accumulators — each source section appends to these.
register_rows = []
parameter_rows = []
benchmark_rows = []
bridge_rows = []
procurement_rows = []

data directory: C:\Users\david\PycharmProjects\night-train-target-network-calib\backend\models\compositions\calib\data


## Source register

One row per source: publication year and **price basis year** are tracked
separately (a 2026 report can quote 2023 prices), plus currency and a
reliability note. Every observation row below carries a `source_id` into this
register.


In [2]:
register_rows += [
    # source_id, short_id, title, publisher, pub_year, price_basis_year,
    # currency, kind, url_or_file, date_accessed, reliability_note
    (
        "S01",
        "nox_model",
        "nox_model.xlsx — night train operator P&L projection model (2025/2030/2036)",
        "TO_VERIFY (Nox Mobility?)",
        "TO_VERIFY",
        "2025-2036 projection",
        "EUR",
        "operator_model",
        "sources/nox_model.xlsx",
        "2026-07-10",
        "Projections not actuals; full cost account structure normalised per "
        "trip / train-km / place-km / % shares; provenance to be confirmed",
    ),
    (
        "S02",
        "uic_nt2",
        "UIC Study Night Trains 2.0",
        "DB International GmbH for UIC",
        "2013",
        "2012",
        "EUR",
        "study",
        "sources/study_night_trains_2.0.pdf",
        "2026-07-10",
        "Cost per available seat-km, Madrid-London cases; high-speed night "
        "train concepts; inflate 2012 -> today before use",
    ),
    (
        "S03",
        "es_valuation",
        "Valuation European Sleeper — Q1 2026",
        "European Sleeper / valuation advisor",
        "2026",
        "2025-2026",
        "EUR",
        "operator_disclosure",
        "sources/6psvratjr74_20260109T10373.pdf",
        "2026-07-10",
        "Real open-access operator; revenue/cost figures and fleet book values",
    ),
    (
        "S04",
        "es_pitch",
        "European Sleeper Sharefunding 2026 pitch deck",
        "European Sleeper",
        "2026",
        "2025",
        "EUR",
        "operator_disclosure",
        "sources/1768215630.pdf",
        "2026-07-10",
        "EUR 10.5M ticket revenue 2025; 3 routes 2026; marketing framing",
    ),
    (
        "S05",
        "oebb_ar25",
        "OeBB annual report 2025",
        "OeBB Holding AG",
        "2026",
        "2025",
        "EUR",
        "operator_disclosure",
        "sources/oebb-annual-report-2025.pdf",
        "2026-07-10",
        "Nightjet fleet/segment data; new-generation trainset procurement",
    ),
    (
        "S06",
        "bmdv_praes",
        "Nachtzugstudie BMDV — Praesentation Ergebnisse",
        "BMDV / consortium",
        "2025",
        "2024-2025",
        "EUR",
        "study",
        "sources/20250331 Nachtzugstudie BMDV Praesentation Erg.pdf",
        "2026-07-10",
        "Revenue requirement 60-250 EUR/place by class; infra prices out of scope here",
    ),
    (
        "S07",
        "bilanz_studie",
        "Studie Bilanz von Nachtzugverkehren (full study behind S06)",
        "Jannis Voll et al. for BMDV",
        "2025",
        "2024-2025",
        "EUR",
        "study",
        "sources/studie-bilanz-nachtzugverkehre.pdf",
        "2026-07-10",
        "Track access / traction energy prices by country (out of scope); "
        "capacity estimates per train type",
    ),
    (
        "S08",
        "ramboll_berlin",
        "Machbarkeitsuntersuchung: Berlin als Drehkreuz eines europaeischen "
        "Nachtzugnetzes",
        "Ramboll for SenUMVK Berlin",
        "2023",
        "2022-2023",
        "EUR",
        "study",
        "sources/berlin-als-drehkreuz-eines-europaeischen-nachtzugnetzes.pdf",
        "2026-07-10",
        "Network feasibility with cost model; German context",
    ),
    (
        "S09",
        "schoenerstedt",
        "Potentialabschaetzung von Nachtzuegen in Europa (Masterarbeit)",
        "Janne Schoenerstedt",
        "2025",
        "2023",
        "EUR",
        "methodology",
        "sources/2025_Janne_Schoenerstedt_Masterarbeit_d.pdf",
        "2026-07-10",
        "Cost build-up follows Standardisierte Bewertung (intraplan/VWI 2023): "
        "capital service per operating hour, maintenance time+km split, "
        "personnel rate build-up — citable official German rate framework",
    ),
    (
        "S10",
        "steer_kcw",
        "Long-distance cross-border passenger rail services",
        "Steer & KCW for European Commission DG MOVE",
        "2021",
        "2020-2021",
        "EUR",
        "study",
        "https://op.europa.eu/en/publication-detail/-/publication/"
        "34244751-6ea3-11ec-9136-01aa75ed71a1",
        "2026-07-10",
        "476pp; night train case studies; rolling stock requirement estimates "
        "(170-250 couchette/sleeper cars for 11 routes); to be extracted",
    ),
    (
        "S11",
        "trafikverket",
        "Trafikverket night train procurement Sweden (TRV 2020/81418)",
        "Trafikverket",
        "2020-2022",
        "2020-2022",
        "SEK",
        "pso_disclosure",
        "https://www.railwaygazette.com/passenger/trafikverket-proposes-"
        "malm%C3%B6-brussels-overnight-service/56392.article",
        "2026-07-10",
        "Estimated subsidy ~60M SEK/yr for ~210k travels gross capacity — "
        "implied cost gap for a real PSO night train; final report ISBN "
        "978-91-8045-099-7; to be extracted",
    ),
    (
        "S12",
        "fr_tet_audit",
        "Intercites de nuit (TET) cost audits / ART reports France",
        "ART / Cour des comptes / DGITM",
        "TO_FILL",
        "TO_FILL",
        "EUR",
        "pso_disclosure",
        "TO_FILL",
        "TO_FILL",
        "State-run night train PSO with published cost per train-km; NOTE "
        "France charges night train TAC at direct cost — normalise before use",
    ),
    (
        "S14",
        "rg_nightjet",
        "OeBB introduces Siemens Nightjet fleet (Matthae interview)",
        "Railway Gazette International",
        "2023",
        "2018-2021 order basis",
        "EUR",
        "press",
        "https://www.railwaygazette.com/passenger/%C3%B6bb-introduces-"
        "siemens-nightjet-fleet-as-paris-berlin-sleeper-train-revived/"
        "65521.article",
        "2026-07-10",
        "OeBB CEO quote: 'more than EUR 700m' invested for 33 new Nightjet "
        "trainsets; also fleet composition (7-car) and delivery plan",
    ),
    (
        "S15",
        "railvolution_nj",
        "OeBB Orders 20 Additional Nightjets (OeBB statement on contract volume)",
        "Railvolution",
        "2021",
        "2021",
        "EUR",
        "press",
        "https://www.railvolution.net/news/obb-orders-20-additional-nightjets",
        "2026-07-10",
        "OeBB statement: second batch of 20 seven-car rakes ~EUR 400M",
    ),
    (
        "S16",
        "cnl_takeover",
        "OeBB 2016 acquisition of DB City Night Line fleet (42 WLABmz sleepers "
        "+ 15 couchette cars for EUR 40M)",
        "press reports (secondary)",
        "2016",
        "2016",
        "EUR",
        "press",
        "TO_FILL (locate primary 2016 report)",
        "2026-07-10",
        "Secondhand acquisition price BEFORE refurbishment — refurb capex "
        "comes on top; secondary sourcing, verify against 2016 primary",
    ),
    (
        "S17",
        "bvwp_update_2024",
        "Aktualisierung der Kosten- und Wertansaetze der "
        "Bundesverkehrswegeplanung (FE VB970452, Schlussbericht Dez 2024)",
        "TTS TRIMODE / Intraplan / Planco / SSP Consult for BMDV",
        "2024",
        "2023-2024",
        "EUR",
        "methodology",
        "https://www.bmv.de/SharedDocs/DE/Anlage/G/aktualisierung-kosten-"
        "wertansaetze-bundesverkehrswegeplanung.pdf?__blob=publicationFile",
        "2026-07-10",
        "Official UPDATE of the BVWP cost rates (supersedes 2016 price basis "
        "used by S09) — resolves the wage-indexation question at the source; "
        "PDF (4MB) not machine-fetchable (bmv.de serves HTML wrapper), manual "
        "download required; TO_EXTRACT: rail vehicle Vorhaltungs-/"
        "Betriebskosten and personnel rates",
    ),
    (
        "S18",
        "sncf_tet_refurb",
        "SNCF Groupe: Au coeur de la production d'un train de nuit "
        "(Intercites de nuit fleet renovation figures)",
        "SNCF Groupe",
        "2023",
        "2021-2023",
        "EUR",
        "operator_disclosure",
        "https://www.groupe-sncf.com/fr/groupe/coulisses/metiers-inattendus/"
        "production-trains-nuit",
        "2026-07-10",
        "Primary operator statement: 129 night coaches deeply renovated for "
        "EUR 91M total, fully state-financed, completed summer 2023 — "
        "0.705 M EUR/coach refurbishment capex; supersedes the ~100M "
        "programme announcements cited by S08/secondary sources",
    ),
    (
        "S13",
        "seed_placeholder",
        "Current seed.py placeholder values (internal)",
        "Back-on-Track target network project",
        "2026",
        "2026",
        "EUR",
        "internal_prior",
        "backend/db/dev/seed.py",
        "2026-07-10",
        "Illustrative placeholders kept as explicit priors so every "
        "replacement is visible in the observation table",
    ),
]

## S13 — Current seed placeholders (explicit priors)

Every parameter starts life here so the calibration notebook runs end-to-end
from day one and every later replacement is a visible diff against a named
prior, never a silent overwrite.

Unit audit finding (must be resolved during calibration):
`coach_maint_eur_km` is documented as €/coach-km in the schema comment, but
`models/evaluation/calc.py` applies it **per train-km** with no coach-count
multiplier. Resolution used downstream: source a per-coach-km rate, emit
`rate × n_coaches` per composition type.


In [3]:
parameter_rows += [
    # source_id, parameter, value, unit, condition, confidence,
    # conversion_note, extraction_note
    ("S13", "driver_costs_eur_h", 52.00, "EUR/h", "any", "low", "", "seed placeholder"),
    ("S13", "crew_costs_eur_h", 38.00, "EUR/h", "any", "low", "", "seed placeholder"),
    ("S13", "driver_overhead_h", 1.0, "h/trip", "any", "low", "", "seed placeholder"),
    ("S13", "crew_overhead_h", 1.0, "h/trip", "any", "low", "", "seed placeholder"),
    (
        "S13",
        "loco_full_service_lease_eur_h",
        145.00,
        "EUR/h",
        "any",
        "low",
        "",
        "seed placeholder",
    ),
    (
        "S13",
        "financing_quota_per",
        0.04,
        "1/year",
        "any",
        "low",
        "",
        "seed placeholder",
    ),
    (
        "S13",
        "fix_overhead_quota_per",
        0.15,
        "share of coach amortisation",
        "any",
        "low",
        "",
        "seed placeholder",
    ),
    (
        "S13",
        "var_overhead_per",
        0.10,
        "share of revenue",
        "any",
        "low",
        "",
        "seed placeholder",
    ),
    (
        "S13",
        "ebit_margin_per",
        0.03,
        "share of revenue",
        "any",
        "low",
        "",
        "seed placeholder",
    ),
    (
        "S13",
        "purchase_coach_eur",
        20000000.00,
        "EUR/coach",
        "any",
        "low",
        "",
        "seed placeholder — implausible; priority replacement",
    ),
    ("S13", "coach_amort_years", 30, "years", "new", "low", "", "seed placeholder"),
    ("S13", "coach_avail_per", 0.80, "share", "any", "low", "", "seed placeholder"),
    (
        "S13",
        "cleaning_eur_day",
        1753.584,
        "EUR/coach/day",
        "any",
        "low",
        "",
        "seed placeholder",
    ),
    (
        "S13",
        "coach_maint_eur_km",
        2.86533333,
        "EUR/train-km (as implemented in calc.py)",
        "any",
        "low",
        "UNIT AUDIT: DDL comment says coach-km but calc.py applies per train-km",
        "seed placeholder",
    ),
    (
        "S13",
        "svc_stockings_eur_place_seat",
        0.01,
        "EUR/place",
        "any",
        "low",
        "",
        "seed placeholder",
    ),
    (
        "S13",
        "svc_stockings_eur_place_couchette",
        0.05,
        "EUR/place",
        "any",
        "low",
        "",
        "seed placeholder",
    ),
    (
        "S13",
        "svc_stockings_eur_place_sleeper",
        0.10,
        "EUR/place",
        "any",
        "low",
        "",
        "seed placeholder",
    ),
]

procurement_rows += [
    # source_id, coach_or_unit, coach_category, condition, price_eur,
    # price_basis, places, year, verify_status, note
    (
        "S13",
        "Current seed placeholder",
        "any",
        "any",
        20000000.00,
        "per coach",
        None,
        2026,
        "placeholder",
        "Known to be implausible — kept as visible prior",
    ),
]

## S01 — nox_model.xlsx (anchor top-down reference)

Complete operator P&L with a cost account structure (210…222) already
normalised per trip, per train-km ("TKM") and per available place-km ("ASK").
2030 column taken as the primary steady-state reference; 2036 kept at total
level. Cell references below are exact.

Bridge caveats carried in `cost_bridge_mapping`: 219 Maintenance includes
loco maintenance (ours sits in the loco lease); 210 Coach Leasing is an
annuity bundling capital + financing (convert before comparing with
amortisation + financing); 211 mixes loco lease and driver cost and must be
split. 212/213/216 are out of scope here.


In [4]:
benchmark_rows += [
    # source_id, scenario, source_account, value_type, value, unit,
    # extraction_note
    (
        "S01",
        "2030 projection",
        "2 - Cost",
        "eur_per_train_km",
        36.598,
        "EUR/train-km",
        "Sheet1 C84",
    ),
    (
        "S01",
        "2030 projection",
        "21 - Route Cost",
        "eur_per_train_km",
        27.097,
        "EUR/train-km",
        "Sheet1 C85",
    ),
    (
        "S01",
        "2030 projection",
        "210 Coach Leasing",
        "eur_per_train_km",
        5.936,
        "EUR/train-km",
        "Sheet1 C86",
    ),
    (
        "S01",
        "2030 projection",
        "211 Loco Leasing & Drivers",
        "eur_per_train_km",
        2.819,
        "EUR/train-km",
        "Sheet1 C87",
    ),
    (
        "S01",
        "2030 projection",
        "212 Track & Station Access",
        "eur_per_train_km",
        3.566,
        "EUR/train-km",
        "Sheet1 C88 — out of scope; kept for total reconciliation",
    ),
    (
        "S01",
        "2030 projection",
        "213 OPS Infrastructure Access",
        "eur_per_train_km",
        0.312,
        "EUR/train-km",
        "Sheet1 C89 — out of scope",
    ),
    (
        "S01",
        "2030 projection",
        "214 Servicing",
        "eur_per_train_km",
        2.634,
        "EUR/train-km",
        "Sheet1 C90",
    ),
    (
        "S01",
        "2030 projection",
        "215 Cabin Crew",
        "eur_per_train_km",
        2.595,
        "EUR/train-km",
        "Sheet1 C91",
    ),
    (
        "S01",
        "2030 projection",
        "216 Electricity",
        "eur_per_train_km",
        2.437,
        "EUR/train-km",
        "Sheet1 C92 — out of scope",
    ),
    (
        "S01",
        "2030 projection",
        "217 Stockings",
        "eur_per_train_km",
        0.320,
        "EUR/train-km",
        "Sheet1 C93",
    ),
    (
        "S01",
        "2030 projection",
        "218 External OPS Services",
        "eur_per_train_km",
        1.136,
        "EUR/train-km",
        "Sheet1 C94",
    ),
    (
        "S01",
        "2030 projection",
        "219 Maintenance",
        "eur_per_train_km",
        5.342,
        "EUR/train-km",
        "Sheet1 C95 — includes loco maintenance",
    ),
    (
        "S01",
        "2030 projection",
        "221 Fixed Overhead Cost",
        "eur_per_train_km",
        4.962,
        "EUR/train-km",
        "Sheet1 C97",
    ),
    (
        "S01",
        "2030 projection",
        "222 Variable Overhead Cost",
        "eur_per_train_km",
        4.540,
        "EUR/train-km",
        "Sheet1 C108",
    ),
    (
        "S01",
        "2030 projection",
        "2 - Cost",
        "ct_per_place_km",
        7.993,
        "EUR-ct/place-km",
        "Sheet1 C123 (ASK basis)",
    ),
    (
        "S01",
        "2030 projection",
        "21 - Route Cost",
        "ct_per_place_km",
        5.918,
        "EUR-ct/place-km",
        "Sheet1 C124",
    ),
    (
        "S01",
        "2030 projection",
        "210 Coach Leasing",
        "ct_per_place_km",
        1.296,
        "EUR-ct/place-km",
        "Sheet1 C125",
    ),
    (
        "S01",
        "2030 projection",
        "215 Cabin Crew",
        "ct_per_place_km",
        0.567,
        "EUR-ct/place-km",
        "Sheet1 C130",
    ),
    (
        "S01",
        "2030 projection",
        "219 Maintenance",
        "ct_per_place_km",
        1.167,
        "EUR-ct/place-km",
        "Sheet1 C134",
    ),
    (
        "S01",
        "2036 projection",
        "2 - Cost",
        "eur_per_train_km",
        34.710,
        "EUR/train-km",
        "Sheet1 D84",
    ),
    (
        "S01",
        "2036 projection",
        "21 - Route Cost",
        "eur_per_train_km",
        29.221,
        "EUR/train-km",
        "Sheet1 D85",
    ),
    (
        "S01",
        "2036 projection",
        "212 Track & Station Access",
        "eur_per_train_km",
        3.405,
        "EUR/train-km",
        "Sheet1 D88 — out of scope",
    ),
    (
        "S01",
        "2036 projection",
        "213 OPS Infrastructure Access",
        "eur_per_train_km",
        0.352,
        "EUR/train-km",
        "Sheet1 D89 — out of scope",
    ),
    (
        "S01",
        "2036 projection",
        "216 Electricity",
        "eur_per_train_km",
        2.683,
        "EUR/train-km",
        "Sheet1 D92 — out of scope",
    ),
    (
        "S01",
        "2036 projection",
        "218 External OPS Services",
        "eur_per_train_km",
        0.955,
        "EUR/train-km",
        "Sheet1 D94",
    ),
    (
        "S01",
        "2036 projection",
        "221 Fixed Overhead Cost",
        "eur_per_train_km",
        1.572,
        "EUR/train-km",
        "Sheet1 D97 — mature-state fixed overhead; quota derivation: "
        "(221+218)=2.53 on in-scope ops base 29.221-3.405-0.352-2.683=22.78 "
        "-> 0.111 share of other operating costs",
    ),
    (
        "S01",
        "2036 projection",
        "222 Variable Overhead Cost",
        "eur_per_train_km",
        3.917,
        "EUR/train-km",
        "Sheet1 D108",
    ),
    (
        "S01",
        "2030 projection",
        "222 Variable Overhead Cost",
        "share_of_revenue",
        0.1428,
        "share",
        "Sheet1 C186 — scale-up state",
    ),
    (
        "S01",
        "2036 projection",
        "222 Variable Overhead Cost",
        "share_of_revenue",
        0.0779,
        "share",
        "Sheet1 D186 — mature state; basis for the var_overhead_per choice",
    ),
    (
        "S01",
        "2030 projection",
        "2 - Cost",
        "eur_per_trip",
        36114.09,
        "EUR/trip",
        "Sheet1 C45",
    ),
    (
        "S01",
        "2030 projection",
        "21 - Route Cost",
        "eur_per_trip",
        26737.92,
        "EUR/trip",
        "Sheet1 C46",
    ),
]

bridge_rows += [
    # source_id, source_account, our_key, in_scope, adjustment_note
    (
        "S01",
        "210 Coach Leasing",
        "coach_amortisation_eur+financing_eur",
        "yes",
        "Lease annuity bundles capital+financing (and possibly heavy "
        "maintenance — verify); convert to purchase-equivalent before "
        "comparing",
    ),
    (
        "S01",
        "211 Loco Leasing & Drivers",
        "loco_eur+driver_eur",
        "yes",
        "Split loco vs driver before per-parameter comparison; split "
        "assumption documented here once made",
    ),
    (
        "S01",
        "212 Track & Station Access",
        "tac_eur+station_charge_eur",
        "no",
        "Infrastructure — excluded from this workstream",
    ),
    (
        "S01",
        "213 OPS Infrastructure Access",
        "shunting_eur+parking_eur",
        "no",
        "Infrastructure — excluded",
    ),
    (
        "S01",
        "214 Servicing",
        "cleaning_eur",
        "yes",
        "Assumed cleaning + service preparation; verify no catering logistics",
    ),
    ("S01", "215 Cabin Crew", "crew_eur", "yes", "Direct match"),
    ("S01", "216 Electricity", "energy_eur", "no", "Energy — separate workstream"),
    ("S01", "217 Stockings", "svc_stockings_eur", "yes", "Direct match"),
    (
        "S01",
        "218 External OPS Services",
        "fix_overhead_eur",
        "partial",
        "Judgment call: external ops services treated as fixed overhead",
    ),
    (
        "S01",
        "219 Maintenance",
        "coach_maintenance_eur",
        "yes",
        "Includes loco maintenance which our model bundles into loco_eur — "
        "deduct an assumed loco share before comparing",
    ),
    (
        "S01",
        "221 Fixed Overhead Cost",
        "fix_overhead_eur",
        "yes",
        "Payroll+tools+offices+marketing+R&D vs fix_overhead_quota on amortisation",
    ),
    (
        "S01",
        "222 Variable Overhead Cost",
        "var_overhead_eur",
        "yes",
        "Incentives+compensations+customer service+payments — matches "
        "%-of-revenue logic",
    ),
]

## S02 — UIC Night Trains 2.0

Cost per available seat-km for the Madrid–London case, 2012 EUR-cents,
published only as totals (incl. infrastructure and energy). Used as a
composition-sensitivity anchor: seat-only vs mixed vs premium-heavy spreads
5.9 → 18.2 ct/place-km — the calibrated model must reproduce that ordering
and rough spread. Inflate 2012 → target year and deduct an assumed
infra+energy share before comparing levels.


In [5]:
benchmark_rows += [
    (
        "S02",
        "Madrid-London 2012",
        "HS Overnight Day Train (500 seats) total",
        "ct_per_place_km",
        5.88,
        "EUR-ct-2012/seat-km",
        "Costs per available seat-km figure — seat-only "
        "lower anchor; total incl. infra+energy",
    ),
    (
        "S02",
        "Madrid-London 2012",
        "HS Simple Night Train (400 seats + 267 berths) total",
        "ct_per_place_km",
        5.29,
        "EUR-ct-2012/seat-km",
        "Same figure",
    ),
    (
        "S02",
        "Madrid-London 2012",
        "HS Traditional Night Train (102 seats + 400 berths + 13 luxury) total",
        "ct_per_place_km",
        6.95,
        "EUR-ct-2012/seat-km",
        "Same figure",
    ),
    (
        "S02",
        "Madrid-London 2012",
        "HS Hotel Night Train (240 luxury beds) total",
        "ct_per_place_km",
        18.21,
        "EUR-ct-2012/seat-km",
        "Same figure — premium-heavy upper anchor",
    ),
]

bridge_rows += [
    (
        "S02",
        "Total cost per ASK (Madrid-London cases)",
        "operator_controllable_subtotal",
        "partial",
        "Published only as total incl. infrastructure and energy; deduct "
        "assumed infra+energy share and inflate 2012 prices before level "
        "comparison; ordering/spread across compositions usable directly",
    ),
]

## S03/S04 — European Sleeper (valuation + pitch deck)

Real open-access operator. Extract here: annual ticket revenue (pitch deck:
EUR 10.5M in 2025), fleet size and book value / refurbishment capex from the
valuation document (→ `coach_procurement_datapoints`, condition
`refurbished`), staffing model if disclosed, cost lines if the valuation
breaks them out.

**TODO (extraction):** work through S03 page by page and append rows below.


In [6]:
benchmark_rows += [
    (
        "S04",
        "FY2025 actual",
        "Ticket revenue",
        "eur_total_year",
        10_500_000,
        "EUR/year",
        "Pitch deck: 'more than EUR 10.5 million in annual ticket "
        "sales' 2025 — revenue side context, not a cost observation",
    ),
]

benchmark_rows += [
    (
        "S03",
        "FY2025",
        "Operating costs",
        "eur_total_year",
        11_900_000,
        "EUR/year",
        "Valuation P&L: opex 11,900 kEUR against 10,500 kEUR revenue",
    ),
    (
        "S03",
        "FY2025",
        "Overhead team costs",
        "eur_total_year",
        819_000,
        "EUR/year",
        "Valuation P&L",
    ),
    (
        "S03",
        "FY2025",
        "Overhead general costs",
        "eur_total_year",
        1_885_000,
        "EUR/year",
        "Valuation P&L — team+general = 2,704 kEUR = 22.7% of "
        "opex / 25.7% of revenue (start-up phase)",
    ),
    (
        "S03",
        "FY2025",
        "EBITDA",
        "eur_total_year",
        -4_104_000,
        "EUR/year",
        "Valuation P&L",
    ),
    (
        "S03",
        "steady-state projection (final year)",
        "Overhead share of operating costs",
        "share_of_cost",
        0.059,
        "share",
        "Final projection column: overheads ~3.66M on opex 61.8M — "
        "mature-phase overhead ratio corridor",
    ),
]

procurement_rows += [
    (
        "S03",
        "European Sleeper coach fleet (refurbished second-hand)",
        "mixed",
        "refurbished",
        None,
        "acquisition+refurb per coach",
        None,
        "2023-2026",
        "TO_VERIFY",
        "Extract fleet book value / capex per coach from valuation document",
    ),
]

## S05 — ÖBB annual report 2025

Extract here: Nightjet new-generation trainset order (count, total volume →
per-coach price, condition `new`), fleet size, any segment cost/revenue
disclosure, depreciation policy for rolling stock (→ `coach_amort_years`,
condition `new`).

Extracted: rolling stock useful life **5–50 years** (AR 2025, p. 212 depreciation table) — the 30-year amortisation assumption for new coaches sits inside official OeBB policy. The AR itself discloses no per-trainset order value; procurement prices come from press disclosures (S14/S15/S16). New-generation Nightjet formation for the composition proposals: fixed 7-car set, ~254 places, 2 sleeping cars + 3 couchette cars with mini-cabins + seating + multifunction car.


In [7]:
parameter_rows += [
    (
        "S05",
        "coach_amort_years",
        "5-50",
        "years (policy range)",
        "new",
        "high",
        "",
        "OeBB AR 2025 p.212: rolling stock useful life 5-50 years — supports "
        "30y for new coaches",
    ),
]

procurement_rows += [
    (
        "S14",
        "Nightjet Viaggio Next Level order, 33 x 7-car trainsets",
        "mixed (2 sleeper + 3 couchette/mini-cabin + seat + multifunction)",
        "new",
        3_030_000,
        "per coach (derived: >700M EUR / 33 trainsets / 7 cars)",
        254,
        2023,
        "verified",
        "OeBB CEO Matthae to Railway Gazette Dec 2023: 'more than EUR 700m' "
        "for 33 trainsets",
    ),
    (
        "S15",
        "Nightjet second batch, 20 x 7-car trainsets",
        "mixed (2 sleeper + 3 couchette/mini-cabin + seat + multifunction)",
        "new",
        2_857_143,
        "per coach (derived: ~400M EUR / 20 / 7)",
        254,
        2021,
        "verified",
        "OeBB statement to Railvolution 2021: batch volume ~400M",
    ),
    (
        "S16",
        "DB City Night Line fleet takeover (42 sleepers + 15 couchette cars)",
        "mixed (sleeper+couchette)",
        "refurbished (acquisition only, pre-refurbishment)",
        701_754,
        "per coach (derived: 40M EUR / 57 cars)",
        None,
        2016,
        "TO_VERIFY",
        "Secondary press sourcing; refurbishment capex additional — locate "
        "primary 2016 disclosure and ES refurb costs (S03) to complete",
    ),
]

## S06/S07 — BMDV night train study

Extract here: revenue requirement per place by class (60–250 EUR range —
useful cross-check on the *sum* of per-place costs), capacity per train
type (validates our standard composition place counts), any operating cost
per train-km statements. Infrastructure/energy price tables are out of
scope.

**TODO (extraction).**


In [8]:
benchmark_rows += [
    (
        "S07",
        "study cost model",
        "Total night train cost",
        "ct_per_place_km",
        6.39,
        "EUR-ct/place-km",
        "Study's own model: 6.39 ct/Platzkm night train vs 5.0-5.5 ct/seat-km "
        "short-haul air — INCLUDES infrastructure and energy; third "
        "independent per-place-km anchor between UIC and nox",
    ),
    (
        "S07",
        "Nelldal 2017 via study",
        "Total cost, 10-coach train, German section",
        "eur_per_train_km",
        25.0,
        "EUR/train-km",
        "Stockholm-Hamburg study cited in S07: ~25 EUR/Zugkm on the German "
        "section (highest TAC) for a 10-coach formation — INCLUDES "
        "infrastructure",
    ),
    (
        "S07",
        "study cost model",
        "Revenue requirement per sold place (Seat)",
        "eur_per_place_trip",
        60.0,
        "EUR/place",
        "'Erloessaetze je verkauften Sitz bzw. Bett zwischen 60 EUR (Sitz) und "
        "250 EUR (Einzelkabine), um wirtschaftlich zu sein' — cross-check on "
        "the SUM of per-place costs",
    ),
    (
        "S07",
        "study cost model",
        "Revenue requirement per sold place (single sleeper cabin)",
        "eur_per_place_trip",
        250.0,
        "EUR/place",
        "Same passage — upper end",
    ),
]

bridge_rows += [
    (
        "S07",
        "Total night train cost (6.39 ct/place-km)",
        "operator_controllable_subtotal",
        "partial",
        "Includes infrastructure (their TAC assumption: 2.76 EUR/train-km) "
        "and energy — strip both before level comparison",
    ),
]

## S08 — Ramboll Berlin-Drehkreuz study

Extract here: the study's operating cost assumptions per train-km or per
train, fleet sizing logic, and any per-category split. **TODO (extraction).**


In [9]:
# S08 findings: the Ramboll study is primarily qualitative on operating
# costs (market/network/regulatory analysis) — no per-unit cost tables to
# extract. One procurement-relevant pointer:
procurement_rows += [
    (
        "S18",
        "French TET night coach deep renovation (129 coaches, completed summer 2023)",
        "mixed (couchette+seat)",
        "refurbished (refurbishment capex only)",
        705_426,
        "per coach (derived: 91M EUR / 129 coaches)",
        None,
        2023,
        "verified",
        "SNCF Groupe primary statement; supersedes the ~100M/130-coach "
        "programme announcements (S08 Ramboll and press); deep renovation of "
        "1980s Corail stock without full rebuild — completes the refurbished "
        "purchase cost: S16 acquisition (~0.70M) + this (~0.71M) ~ 1.4M/coach",
    ),
]

## S09 — Schönerstedt thesis / Standardisierte Bewertung (intraplan/VWI 2023)

Primary bottom-up rate framework: capital service per operating hour,
maintenance split into time- and km-dependent components, personnel rate
build-up (wage + surcharges → cost per real deployment hour — exactly our
billable-hours logic incl. overhead). Extract the concrete rate tables the
thesis applies and cite them as *Standardisierte Bewertung 2023* values.

**TODO (extraction):** thesis appendix parameter tables → `parameter_rows`
for driver/crew rates, maintenance components, capital service.


In [10]:
parameter_rows += [
    (
        "S09",
        "driver_costs_eur_h",
        57.0,
        "EUR/h",
        "any",
        "medium",
        "Price basis BVWP 2016 — consider wage indexation to target year",
        "Thesis parameter table (Gehalt Lokfuehrer), source PTV et al. 2016 "
        "(Methodikhandbuch Bundesverkehrswegeplan)",
    ),
    (
        "S09",
        "trainmanager_costs_eur_h",
        49.0,
        "EUR/h",
        "any",
        "medium",
        "BVWP 2016 basis",
        "Gehalt Zugfuehrer — reflect via coach crew "
        "factors if a train manager is carried",
    ),
    (
        "S09",
        "crew_costs_eur_h",
        39.0,
        "EUR/h",
        "any",
        "medium",
        "BVWP 2016 basis",
        "Gehalt Zugbegleiter",
    ),
    (
        "S09",
        "coach_capital_service_eur_h",
        18.72,
        "EUR/operating-hour",
        "new",
        "medium",
        "BVWP rate +1/3 night-train uplift applied by thesis",
        "Tabelle 10 invest_cost_hour 0.01872 Tsd EUR/h, all four coach types "
        "— convertible to purchase-equivalent via operating hours and annuity",
    ),
    (
        "S09",
        "coach_maint_eur_km",
        3.25,
        "EUR/coach-km",
        "any",
        "medium",
        "BVWP rate +1/3 night-train uplift",
        "Tabelle 10 maintenance_cost_km 0.00325 Tsd EUR/km. VERIFIED AGAINST "
        "S17 (primary, Tab. 7-5): REFUTED as a coach-level rate — the BVWP "
        "tables are per TRAIN-km for full trainsets (1.3-7.8 EUR/train-km); "
        "a 190m 230km/h multi-system train costs 5.96 EUR/train-km, i.e. "
        "~0.80 EUR/coach-km. The thesis appears to have read a per-train "
        "value as per-coach. Superseded by the S17 observations",
    ),
    (
        "S09",
        "loco_capital_service_eur_h",
        120.0,
        "EUR/h",
        "any",
        "medium",
        "BVWP, 'leicht erhoeht'",
        "Loco table invest_cost_hour 0.12 Tsd EUR/h",
    ),
    (
        "S09",
        "loco_maint_eur_km",
        2.0,
        "EUR/km",
        "any",
        "medium",
        "BVWP",
        "Loco table maintenance_cost_km — capital + maintenance together form "
        "a corridor for our bundled full-service lease rate",
    ),
    (
        "S09",
        "svc_stockings_eur_place",
        15.0,
        "EUR (unit ambiguous — likely per served bed/trip)",
        "any",
        "low",
        "",
        "Cservice,nighttrain 0.015 Tsd EUR 'z.B. Bettwaesche' — unit not fully "
        "specified; conflicts with nox-derived ~0.7 EUR/place avg; likely "
        "includes breakfast/catering",
    ),
    (
        "S09",
        "crew_factor_per_coach",
        1.0,
        "attendants/coach",
        "any",
        "medium",
        "",
        "Tabelle 10 train_attendant_count = 1 per coach — upper corridor vs our 0.5",
    ),
]

# Composition-informing specs from Tabelle 10 (not cost parameters):
# all coach types 50 t, 160 km/h; places: seat 70, couchette 36,
# sleeper 20, capsule 40 — used as cross-check in 02 section 3.

## S10/S11/S12 — Steer/KCW, Trafikverket, French TET (web sources)

- **S10:** case-study cost assumptions and rolling stock cost estimates.
- **S11:** subsidy ≈ 60M SEK/yr for ~210k travels of gross capacity —
  consistency check on the *net* result of a real PSO night train (subsidy =
  cost − revenue), not a gross cost level. Convert SEK→EUR at
  contract-period rates.
- **S12:** locate ART / Cour des comptes cost-per-train-km disclosures;
  remember French night train TAC is set at direct cost — strip before use.

**TODO (extraction).**


In [11]:
bridge_rows += [
    (
        "S11",
        "Annual subsidy for defined capacity",
        "operator_controllable_subtotal",
        "partial",
        "Subsidy = cost minus revenue; consistency check on net result, not "
        "gross cost level; SEK->EUR at contract-period rate",
    ),
]

bridge_rows += [
    (
        "S13",
        "seed placeholders (all parameters)",
        "per-parameter",
        "yes",
        "Explicit priors; every calibrated value replaces one of these rows visibly",
    ),
]

## Write the observation CSVs


## S17 — BVWP cost rate update Dec 2024 (primary, uploaded)

*Aktualisierung der Kosten- und Wertansätze der Bundesverkehrswegeplanung*,
Schlussbericht Dezember 2024 (FE VB970452), price basis **2021** (vehicle
costs escalated from 2012 by +8.3% via Erzeugerpreisindex Schienenfahrzeuge,
Destatis GP19-30202). This is the official successor to the 2016 rates the
thesis (S09) used — it resolves both the maintenance-unit conflict and the
wage-indexation question at the source.

Key structural notes:
- Personnel rates (Tab. 7-1/7-2, p. 87–88) are per **deployment hour** with
  roster efficiency (Dienstplanwirkungsgrad 60%/70%) already embedded —
  standby, prep/wrap-up, training, positioning. Using them together with a
  separate per-trip overhead allowance would double-count.
- Vehicle rates (Tab. 7-4/7-5, p. 90–91) are per **Modellfahrzeug =
  trainset**, not per coach; maintenance rates include shares for shunting
  and depot runs.
- *"Für Nacht- und Autoreisezüge wurden keine Modellfahrzeugtypen
  definiert"* — no night train vehicle types exist in the source; any
  night-train uplift is a modelling assumption on top.
- Tab. 7-6 holds official traction energy consumption rates per vehicle
  type — out of scope here, but directly relevant to the **energy model
  workstream** (noted in the README).


In [12]:
parameter_rows += [
    (
        "S17",
        "driver_costs_eur_h",
        68.30,
        "EUR/deployment-hour",
        "any",
        "high",
        "2021 tariff basis (BuRa-ZugTV/GDL LF5); build-up: +20% employer "
        "social, +15% night/Sunday, +15% admin overhead, +2% pension "
        "(factor 1.619), 1648 h/yr, Dienstplanwirkungsgrad 60%",
        "Tab. 7-2 p.88 Triebfahrzeugfuehrer — roster efficiency embedded; "
        "raw hourly rate 40.95 EUR/h",
    ),
    (
        "S17",
        "trainmanager_costs_eur_h",
        62.70,
        "EUR/deployment-hour",
        "any",
        "high",
        "Same build-up; +10% equipment/uniform; 70% roster efficiency",
        "Tab. 7-2 Zugchef SPV — raw 43.91 EUR/h",
    ),
    (
        "S17",
        "crew_costs_eur_h",
        51.30,
        "EUR/deployment-hour",
        "any",
        "high",
        "Same build-up; +10% equipment/uniform; 70% roster efficiency",
        "Tab. 7-2 Zugbetreuer SPV — raw 35.90 EUR/h",
    ),
    (
        "S17",
        "crew_per_train",
        "1 driver + 1 Zugchef + 1-3 Zugbetreuer",
        "persons/train",
        "any",
        "high",
        "",
        "Tab. 7-7 p.93 — SPFV day trains; night trains need own assumption",
    ),
    (
        "S17",
        "coach_amort_years",
        30,
        "years",
        "new",
        "high",
        "",
        "Tab. 7-4 p.90 Nutzungsdauer der SPFV-Zuege — confirms S05 policy "
        "range and the 30y choice",
    ),
    (
        "S17",
        "coach_avail_per",
        0.909,
        "share",
        "new",
        "high",
        "Derived: 10% Betriebs- und Werkstattreserve -> 1/1.1",
        "Tab. 7-4 — official reserve assumption; brutto deployment "
        "4380 h/yr (365d x 12h)",
    ),
    (
        "S17",
        "redesign_cost_share",
        0.15,
        "share of initial investment, at year 15",
        "new",
        "high",
        "",
        "Tab. 7-4 — mid-life redesign; NOT covered by our straight-line "
        "purchase amortisation — documented in README",
    ),
    (
        "S17",
        "real_interest_rate",
        0.017,
        "1/year (real)",
        "any",
        "high",
        "",
        "Tab. 7-4 massgebender Realzinssatz — societal discount rate, "
        "NOT an operator WACC; recorded as lower bound context for "
        "financing_quota",
    ),
    (
        "S17",
        "coach_maint_eur_km",
        0.80,
        "EUR/coach-km (derived)",
        "new",
        "high",
        "Derived: HGV C multi-system (190m, 390 seats, 230 km/h) "
        "5.96 EUR/train-km / ~7.5 vehicle units; rates include shunting/"
        "depot-run shares; price basis 2021",
        "Tab. 7-5 p.91 — settles the S09 conflict: BVWP maintenance is "
        "per train-km, ~0.80 EUR/coach-km at night-train-like formations",
    ),
]

procurement_rows += [
    (
        "S17",
        "HGV C reference trainset (190m, 390 seats, 230 km/h, multi-system)",
        "mixed trainset (HGV C reference)",
        "new",
        3_575_714,
        "per coach-equivalent (derived: 25.03 M EUR / 7 units)",
        390,
        2021,
        "verified",
        "Tab. 7-5 investment 25.03 M EUR multi-system — third independent "
        "new-build anchor, consistent with Nightjet-derived 2.86-3.03 M",
    ),
]

## 2026-07 verification round — sources S19–S41

Appended by the July 2026 calibration sessions: wage-chain sources, cross-country
verification, loco-market accounts, ES/Italo financials, coach procurement
contracts, and the BoT route-geometry sample. Full derivations in CALIBRATION.md;
this cell keeps the register and the machine-readable observations in sync.


In [13]:
# --- register (S19–S41; S31/S32 were duplicates of S04/S03 and are not re-issued) ---
_R = [
    (
        "S19",
        "db_evg_2023",
        "DB–EVG Tarifabschluss 2023 (flat-rate steps)",
        "DB/EVG via press",
        2023,
        2023,
        "EUR",
        "wage agreement",
        "press",
        "2026-07",
        "official outcome via press",
    ),
    (
        "S20",
        "db_tarif_2025",
        "DB-Tarifrunde 2025 (percentage steps)",
        "DB/EVG via press",
        2025,
        2025,
        "EUR",
        "wage agreement",
        "press",
        "2026-07",
        "official outcome via press",
    ),
    (
        "S21",
        "ecb_dec_2025",
        "ECB staff projections Dec 2025 (wage growth 2028+)",
        "ECB",
        2025,
        None,
        "EUR",
        "macro projection",
        "ecb.europa.eu",
        "2026-07",
        "assumption base for 2028–32",
    ),
    (
        "S22",
        "fr_driver_aggr",
        "France driver salary aggregators",
        "aggregators",
        2026,
        2025,
        "EUR",
        "salary data",
        "web",
        "2026-07",
        "low — no official rate card",
    ),
    (
        "S23",
        "it_driver_aggr",
        "Italy driver salary aggregators",
        "aggregators",
        2026,
        2025,
        "EUR",
        "salary data",
        "web",
        "2026-07",
        "low — no official rate card",
    ),
    (
        "S24",
        "sj_payscale",
        "SJ AB official pay scale (Seko agreement)",
        "SJ/Seko",
        2025,
        2025,
        "SEK",
        "official pay scale",
        "web",
        "2026-07",
        "medium-high",
    ),
    (
        "S25",
        "oebb_careers",
        "ÖBB careers pages (driver)",
        "ÖBB",
        2026,
        2026,
        "EUR",
        "official careers",
        "oebb.at",
        "2026-07",
        "high",
    ),
    (
        "S26",
        "fr_crew_aggr",
        "France chef de bord aggregators",
        "aggregators",
        2026,
        2025,
        "EUR",
        "salary data",
        "web",
        "2026-07",
        "low",
    ),
    (
        "S27",
        "it_crew_aggr",
        "Italy capotreno aggregators",
        "aggregators",
        2026,
        2025,
        "EUR",
        "salary data",
        "web",
        "2026-07",
        "low",
    ),
    (
        "S28",
        "oebb_zub",
        "ÖBB Karriere Zugbegleiter:in",
        "ÖBB",
        2026,
        2026,
        "EUR",
        "official careers",
        "oebb.at",
        "2026-07",
        "high; incl. allowances, 14 salaries",
    ),
    (
        "S29",
        "ell_ja2021",
        "ELL GmbH & Co. KG Jahresabschluss 2021",
        "Bundesanzeiger",
        2022,
        2021,
        "EUR",
        "audited accounts",
        "bundesanzeiger.de",
        "2026-07",
        "audited; internal-rate caveats",
    ),
    (
        "S30",
        "rdc_ja2021",
        "RDC Deutschland Jahresabschluss 2021",
        "Bundesanzeiger",
        2022,
        2021,
        "EUR",
        "audited accounts",
        "bundesanzeiger.de",
        "2026-07",
        "checked-empty for rates",
    ),
    (
        "S33",
        "italo_2023",
        "Italo/NTV 2022–23 figures via financial press",
        "press",
        2024,
        2023,
        "EUR",
        "financial press",
        "web",
        "2026-07",
        "actuals, medium-high",
    ),
    (
        "S34",
        "cd_tenders",
        "ČD loco tender figures (2021 lease framework; 2026 dual-mode bid; ComfortJet)",
        "ČD/Siemens via zdopravy.cz",
        2026,
        2026,
        "CZK",
        "tender figures",
        "web",
        "2026-07",
        "estimates and bids",
    ),
    (
        "S35",
        "italo_ar2019",
        "Italo S.p.A. Annual Report 2019 (EN)",
        "Italo/NTV",
        2020,
        2019,
        "EUR",
        "audited accounts",
        "sources/financial-statement-2019.pdf",
        "2026-07",
        "audited primary",
    ),
    (
        "S36",
        "trafikverket_talgo",
        "Trafikverket night train procurement — EXTENDS S11 (2025 failed round; 2026 Talgo award; SJ refit)",
        "Trafikverket/SJ via Swedish press",
        2026,
        2026,
        "SEK",
        "procurement",
        "web",
        "2026-07",
        "award reported; scope caveats",
    ),
    (
        "S37",
        "norske_flirtnex",
        "Norske tog FLIRTNEX 17×8-car NOK 8bn (2023)",
        "Norske tog",
        2023,
        2023,
        "NOK",
        "contract",
        "web",
        "2026-07",
        "signed",
    ),
    (
        "S38",
        "calsleeper_mk5",
        "Caledonian Sleeper Mk5 75 cars £150M (2015)",
        "Serco/CAF via press",
        2015,
        2015,
        "GBP",
        "contract",
        "web",
        "2026-07",
        "delivered",
    ),
    (
        "S39",
        "skoda_icn",
        "Trenitalia ICN framework ≤370 cars €732.5M; call 70/€138.6M (2023)",
        "Trenitalia/Škoda; BoT analysis",
        2023,
        2023,
        "EUR",
        "contract",
        "web",
        "2026-07",
        "signed; budget-capped",
    ),
    (
        "S40",
        "gebrauchtzug",
        "DB Gebrauchtzug channel evidence (Bieterverfahren; 73 multi-voltage coaches offered)",
        "DB Regio",
        2026,
        None,
        "EUR",
        "market channel",
        "db-gebrauchtzug.de",
        "2026-07",
        "no public prices; CNL price under S16",
    ),
    (
        "S41",
        "bot_route_db",
        "BoT Open Night Train Database — route geometry sample",
        "Back-on-Track Europe",
        2026,
        None,
        "n/a",
        "route database",
        "back-on-track.eu/night-train-database",
        "2026-07",
        "2026 timetable",
    ),
]
register_rows.extend(_R)

# --- S06 half-train benchmark (mined 2026-07-20 from the presentation) ---
benchmark_rows.extend(
    [
        (
            "S06",
            "halftrain_base",
            "total",
            "cost_per_train_km",
            44.51,
            "EUR/train-km",
            "348 places; 900 km base; 2 WL×30 + 4 Bc×54 + 1 seat×72; invest 25.05M",
        ),
        (
            "S06",
            "halftrain_base",
            "total",
            "cost_per_place_km",
            6.39,
            "ct/place-km",
            "incl. infra+energy",
        ),
        (
            "S06",
            "halftrain_base",
            "operator_only",
            "cost_per_train_km",
            37.40,
            "EUR/train-km",
            "x0.84 after Energie 9% + Trasse/Station 7%; implied ~150-180k km/consist-yr",
        ),
        (
            "S06",
            "halftrain_base",
            "shares",
            "cost_shares",
            0,
            "percent",
            "Rollmaterial 45 / Reinigung+Abstellung 14 / Overhead 11 / Personal 11 / Energie 9 / Trasse 7",
        ),
    ]
)
parameter_rows.extend(
    [
        (
            "S06",
            "financing_wacc",
            6.5,
            "percent",
            "",
            "medium",
            "full annuity over 32y",
            "study model input",
        ),
        ("S06", "amort_years", 32, "years", "new", "medium", "", "study model input"),
        (
            "S06",
            "driver_eur_h",
            "62-104",
            "EUR/h",
            "15-country range",
            "medium",
            "productive-hour basis",
            "reconciles with 90.33 deployment-h via roster efficiency",
        ),
        (
            "S06",
            "crew_eur_h",
            "Zf 45-70; Betr. 32-47",
            "EUR/h",
            "15-country range",
            "medium",
            "productive-hour basis",
            "47/0.70 ≈ 67 ≈ attendant deployment rate",
        ),
        (
            "S06",
            "coach_price_per_class",
            "WL 3.8 / Bc 2.8 / B 2.0",
            "MEUR",
            "new",
            "medium",
            "144/106/76 kEUR/m",
            "sourced class spread; basis if class factors reintroduced",
        ),
    ]
)
procurement_rows.extend(
    [
        (
            "S36",
            "Talgo night coach",
            "mixed",
            "new",
            4.75e6,
            "2026 award (scope caveats)",
            None,
            2026,
            "TO_VERIFY",
            "excluded from per-metre fit: ~357 kEUR/m, scope unresolvable",
        ),
        (
            "S37",
            "FLIRTNEX car (coach-equivalent)",
            "mixed",
            "new",
            4.0e6,
            "2023 contract, traction share removed",
            None,
            2023,
            "ok",
            "~145 kEUR/m",
        ),
        (
            "S38",
            "CalSleeper Mk5 car",
            "mixed",
            "new",
            2.7e6,
            "2015 contract",
            None,
            2015,
            "ok",
            "117 kEUR/m 2015 -> ~141 escalated",
        ),
        (
            "S39",
            "Skoda ICN sleeper",
            "sleeper",
            "new",
            1.98e6,
            "2023 contract, budget-capped",
            None,
            2023,
            "ok",
            "75 kEUR/m floor",
        ),
        (
            "S06",
            "French rental coach (all-in TCO)",
            "mixed",
            "new",
            678000,
            "per coach-year, 15y rental, 1.83bn/180",
            None,
            2025,
            "TO_VERIFY",
            "upper bracket for annual capital+maintenance",
        ),
    ]
)
print(
    f"appended: {len(_R)} register rows + S06 benchmark/parameter/procurement observations"
)

appended: 21 register rows + S06 benchmark/parameter/procurement observations


In [14]:
def _write(rows, columns, filename):
    df = pd.DataFrame(rows, columns=columns)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    df.to_csv(DATA_DIR / filename, index=False)
    print(f"{filename}: {len(df)} rows")
    return df


register_df = _write(
    register_rows,
    [
        "source_id",
        "short_id",
        "title",
        "publisher",
        "pub_year",
        "price_basis_year",
        "currency",
        "kind",
        "url_or_file",
        "date_accessed",
        "reliability_note",
    ],
    "sources_register.csv",
)
parameter_df = _write(
    parameter_rows,
    [
        "source_id",
        "parameter",
        "value",
        "unit",
        "condition",
        "confidence",
        "conversion_note",
        "extraction_note",
    ],
    "parameter_observations.csv",
)
benchmark_df = _write(
    benchmark_rows,
    [
        "source_id",
        "scenario",
        "source_account",
        "value_type",
        "value",
        "unit",
        "extraction_note",
    ],
    "benchmark_observations.csv",
)
bridge_df = _write(
    bridge_rows,
    ["source_id", "source_account", "our_key", "in_scope", "adjustment_note"],
    "cost_bridge_mapping.csv",
)
procurement_df = _write(
    procurement_rows,
    [
        "source_id",
        "coach_or_unit",
        "coach_category",
        "condition",
        "price_eur",
        "price_basis",
        "places",
        "year",
        "verify_status",
        "note",
    ],
    "coach_procurement_datapoints.csv",
)

# Referential integrity: every observation must point at a registered source.
_known = set(register_df["source_id"])
for name, df in [
    ("parameter", parameter_df),
    ("benchmark", benchmark_df),
    ("bridge", bridge_df),
    ("procurement", procurement_df),
]:
    unknown = set(df["source_id"]) - _known
    assert not unknown, (
        f"{name}_observations references unregistered sources: {unknown}"
    )
print("source_id integrity OK")

sources_register.csv: 39 rows
parameter_observations.csv: 41 rows
benchmark_observations.csv: 49 rows
cost_bridge_mapping.csv: 16 rows
coach_procurement_datapoints.csv: 12 rows
source_id integrity OK
